# 02 – IT Service Desk Analysis
SLA, resolution time, backlog, technician, category, department, CSAT, reopen and trend analysis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
t=pd.read_csv(ROOT/'data'/'processed'/'02_cleaned_tickets.csv',parse_dates=['Created_Timestamp','Resolved_Timestamp'])
tech=pd.read_csv(ROOT/'data'/'processed'/'technicians.csv')
t.head()

In [ ]:
kpi={'Total Tickets':len(t),'SLA Compliance %':100*(t.loc[t.SLA_Status.isin(['Met','Breached']),'SLA_Status']=='Met').mean(),'Average Resolution':t.Resolution_Time_Hours.mean(),'Average CSAT':t.Customer_Satisfaction.mean()}
kpi

In [ ]:
sla_by_priority=t.loc[t.SLA_Status.isin(['Met','Breached'])].groupby('Priority').agg(Tickets=('Ticket_ID','count'),Breaches=('SLA_Breached','sum'),Avg_Resolution=('Resolution_Time_Hours','mean'))
sla_by_priority['SLA_Compliance_%']=100*(1-sla_by_priority['Breaches']/sla_by_priority['Tickets'])
sla_by_priority.sort_values('SLA_Compliance_%')

In [ ]:
tech_perf=t.groupby(['Technician_ID','Assigned_Technician']).agg(Tickets_Assigned=('Ticket_ID','count'),Avg_Resolution=('Resolution_Time_Hours','mean'),Avg_Response=('First_Response_Minutes','mean'),SLA_Breaches=('SLA_Breached','sum'),Avg_CSAT=('Customer_Satisfaction','mean')).reset_index()
tech_perf['Workload_Rank']=tech_perf['Tickets_Assigned'].rank(method='dense',ascending=False)
tech_perf.sort_values('Tickets_Assigned',ascending=False).head(10)

In [ ]:
active=t['Status'].isin(['Open','Assigned','In Progress','Pending','Reopened'])
backlog=t.loc[active]
pd.pivot_table(backlog,index='Ticket_Age_Band',columns='Priority',values='Ticket_ID',aggfunc='count',fill_value=0)

In [ ]:
monthly=t.set_index('Created_Timestamp').resample('MS').agg(Tickets=('Ticket_ID','count'),SLA_Breaches=('SLA_Breached','sum'),Avg_Resolution=('Resolution_Time_Hours','mean'))
monthly['MoM_%']=monthly['Tickets'].pct_change()*100
monthly['YoY_%']=monthly['Tickets'].pct_change(12)*100
monthly['Rolling_3M']=monthly['Tickets'].rolling(3,min_periods=1).mean()
monthly['Previous_Month']=monthly['Tickets'].shift(1)
monthly.tail(12)

In [ ]:
category=t.groupby('Category').agg(Tickets=('Ticket_ID','count'),Breaches=('SLA_Breached','sum'),Avg_Resolution=('Resolution_Time_Hours','mean')).sort_values('Tickets',ascending=False)
category['Share_%']=100*category['Tickets']/category['Tickets'].sum()
category['Cumulative_%']=category['Share_%'].cumsum()
category

In [ ]:
# CSAT vs resolution time: association, not causation.
assoc=t[['Resolution_Time_Hours','Customer_Satisfaction']].dropna()
assoc.corr()

In [ ]:
# Reopen analysis
reopen=t.groupby('Category').agg(Tickets=('Ticket_ID','count'),Reopened=('Reopened',lambda x:(x=='Yes').sum()))
reopen['Reopen_Rate_%']=100*reopen['Reopened']/reopen['Tickets']
reopen.sort_values('Reopen_Rate_%',ascending=False)

In [ ]:
# loc/iloc, value_counts, apply/map support practical drill-downs.
critical=t.loc[t['Priority'].eq('Critical'),['Ticket_ID','Category','Status','Resolution_Time_Hours','SLA_Status']]
status_counts=t['Status'].value_counts()
priority_weight={'Critical':4,'High':3,'Medium':2,'Low':1}
t['Priority_Weight']=t['Priority'].map(priority_weight)
t['Needs_Review']=t.apply(lambda r:'Yes' if r['SLA_Breached']==1 and r['Priority'] in ['Critical','High'] else 'No',axis=1)
critical.iloc[:10], status_counts